# Scraper de géneros musicales
Scraping de géneros desde Last.fm a partir del dataset de letras.


In [ ]:
# Ejecuta esta celda solo la primera vez
!pip install cloudscraper beautifulsoup4 pandas requests -q


## Imports y configuración


In [ ]:
import re
import time
import random
import logging
import pandas as pd
import cloudscraper
from bs4 import BeautifulSoup
from urllib.parse import quote

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

SCRAPER = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)

HEADERS_LASTFM = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

LASTFM_DELAY = (1, 3)

print("Configuración cargada")


## Funciones de scraping


In [ ]:
def _get_lastfm(url: str):
    time.sleep(random.uniform(*LASTFM_DELAY))
    try:
        resp = SCRAPER.get(url, headers=HEADERS_LASTFM, timeout=15)
        resp.raise_for_status()
        return resp
    except Exception as e:
        log.warning(f"Last.fm request failed: {url} → {e}")
        return None

def _extract_lastfm_tags(html: str, top_n: int):
    soup = BeautifulSoup(html, "html.parser")
    tags = []
    for selector in [".catalogue-tags .tag", ".catalogue-tags a",
                     ".tag-list .tag", ".tags-list a", "ul.tags li a", "a.tag"]:
        for el in soup.select(selector):
            tag = el.get_text(strip=True).lower()
            if tag and tag not in tags and len(tag) > 1:
                tags.append(tag)
        if tags:
            break
    if not tags:
        seen = set()
        for a in soup.find_all("a", href=True):
            if "/tag/" in a["href"]:
                tag = a.get_text(strip=True).lower()
                if tag and tag not in seen and len(tag) > 1:
                    tags.append(tag); seen.add(tag)
    return tags[:top_n]

def scrape_genre(artist: str, song: str = None, top_n: int = 3):
    tags = []
    if song:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}/_/{quote(song)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    if not tags:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    log.info(f"Géneros: {tags}")
    return tags

def scrape_genres_dataset(df: pd.DataFrame, output_csv: str = "genres_dataset.csv") -> pd.DataFrame:
    results = []
    # Cargar progreso previo si existe
    try:
        done_df = pd.read_csv(output_csv)
        done_keys = set(zip(done_df["ALink"].str.lower(), done_df["SName"].str.lower()))
        log.info(f"Reanudando: {len(done_df)} canciones ya procesadas")
        results = done_df.to_dict("records")
    except FileNotFoundError:
        done_keys = set()

    pending = df[~df.apply(lambda r: (str(r["ALink"]).lower(), str(r["SName"]).lower()) in done_keys, axis=1)]
    log.info(f"Canciones pendientes: {len(pending)}")

    for i, (_, row) in enumerate(pending.iterrows(), 1):
        artist = row["ALink"].strip("/").split("/")[-1].replace("-", " ")
        song   = row["SName"]
        log.info(f"[{i}/{len(pending)}] {artist} — {song}")

        genres = scrape_genre(artist, song, top_n=3)

        results.append({
            "ALink":   row["ALink"],
            "SName":   song,
            "Lyric":   row["Lyric"],
            "genres":  ", ".join(genres) if genres else None,
            "genre_1": genres[0] if len(genres) > 0 else None,
            "genre_2": genres[1] if len(genres) > 1 else None,
            "genre_3": genres[2] if len(genres) > 2 else None,
        })
        pd.DataFrame(results).to_csv(output_csv, index=False, encoding="utf-8")

    log.info(f"Dataset guardado: {output_csv} ({len(results)} canciones)")
    return pd.DataFrame(results)

print("Funciones cargadas")


## Cargar dataset


In [ ]:
# Ruta al dataset generado en Kaggle
INPUT_CSV = "/kaggle/working/letras_en_kaggle.csv"

df = pd.read_csv(INPUT_CSV)

# Rango de canciones a scrapear (índices basados en 1, ambos incluidos)
# Cambia START y END según el tramo que quieras procesar
START = 1       # primera canción a scrapear
END   = None    # última canción a scrapear (None = hasta el final)

df_range = df.iloc[START - 1 : END]

print(f"Total canciones en dataset: {len(df)}")
print(f"Rango seleccionado:         {START} → {END or len(df)}")
print(f"Canciones a scrapear:       {len(df_range)}")
df_range.head(3)


## Ejecutar scraping


In [ ]:
OUTPUT_CSV = f"genres_dataset_{START}_{END}.csv"

df_result = scrape_genres_dataset(df_range, output_csv=OUTPUT_CSV)

print(f"Total canciones:  {len(df_result)}")
print(f"Con géneros:      {df_result['genres'].notna().sum()} ({df_result['genres'].notna().mean():.0%})")


## Explorar resultados


In [ ]:
df_result = pd.read_csv(OUTPUT_CSV)

print(f"Total canciones:  {len(df_result)}")
print(f"Con géneros:      {df_result['genres'].notna().sum()} ({df_result['genres'].notna().mean():.0%})")
print(f"\nGéneros más frecuentes:")
print(df_result["genre_1"].value_counts().head(10).to_string())


In [ ]:
df_result[["ALink", "SName", "genres"]].head(10)
